# 🎮 Drug Ranking Tool — Interactive Widget Demo

> Tool A — 互動式藥物推薦 Widget

選擇 cancer type + cell line → **即時** 顯示 Top K 推薦藥物與 95% CI。

## 開始之前

**在 Colab 上**:點 `Runtime → Run all`,直接跑完即可看到 Widget。

**在本機**:
1. 已安裝 `requirements.txt`(`pip install -r requirements.txt`)
2. 已下載 `intermediates/` 目錄(GitHub Release 或執行 notebooks 01–05)
3. `jupyter notebook notebooks/06_drug_ranking_widget.ipynb`

## Cell 1 — 環境準備

In [ ]:
# 偵測執行環境(Colab vs 本機)
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    # === Colab 設定:clone repo + 安裝套件 + 下載 intermediates ===
    GITHUB_USER = 'otonifrio2812'  # (已預填您的帳號)
    REPO_NAME = 'pediatric-bt-drug-prediction'
    RELEASE_TAG = 'v1.0.1'  # ← 更新此處發新版時

    import os
    if not os.path.exists(REPO_NAME):
        !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

    # 安裝套件(含 NumPy<2 來避免 pickle 二進位相容問題)
    !pip install -q -r requirements.txt

    # 下載 intermediate files(用 wget 從 GitHub Release 抓)
    if not os.path.exists('intermediates/stage6_ensemble_models.pkl'):
        print('Downloading intermediates from GitHub Release...')
        os.makedirs('intermediates', exist_ok=True)
        RELEASE_URL = (
            f'https://github.com/{GITHUB_USER}/{REPO_NAME}/'
            f'releases/download/{RELEASE_TAG}/intermediates.zip'
        )
        !wget -q {RELEASE_URL} -O intermediates.zip
        !unzip -q intermediates.zip -d intermediates/
        print('✓ Intermediates ready')

    # ⚠️ Colab 預設 NumPy 2.x 與我們 pickle 中繼檔(NumPy 1.x 序列化)不相容
    # 強制降版到 <2.0,需要 RESTART RUNTIME 才會生效
    print('\n📦 Checking NumPy version for binary compatibility...')
    import numpy as np
    if np.__version__.startswith('2'):
        print(f'   Current numpy={np.__version__}, downgrading to <2.0...')
        !pip install -q 'numpy<2.0' 'scipy<1.13'
        print('\n' + '='*60)
        print('⚠️  ACTION REQUIRED:')
        print('    Click: Runtime → Restart session')
        print('    Then run this Cell 1 again (will skip downloads).')
        print('    Then continue to Cell 2.')
        print('='*60)
    else:
        print(f'   ✓ numpy={np.__version__} is compatible')
else:
    # === 本機:確認在 repo 根目錄 ===
    import os
    if not os.path.exists('src'):
        os.chdir('..')  # notebook 在 notebooks/ 裡,往上跳一層
    print(f'Working directory: {os.getcwd()}')

# 把 src 加進 Python path
sys.path.insert(0, '.')

## Cell 2 — 載入模型與資料

In [ ]:
from src.drug_ranking import load_artifacts, list_cells_by_cancer_type

print('Loading artifacts (預計 30 秒)...')
artifacts = load_artifacts(intermediates_dir='intermediates/')

cells_by_type = list_cells_by_cancer_type(artifacts)
print(f'\n✓ Loaded:')
print(f'  • {len(artifacts.cell_metadata_lookup)} cell lines')
print(f'  • {len(artifacts.drug_text_lookup)} drugs (with target/pathway annotations)')
print(f'  • {len(artifacts.ensemble)} ensemble models')
print(f'  • Cancer types: {sorted(cells_by_type.keys())[:5]}...')
print(f'  • Total cancer types: {len(cells_by_type)}')

## Cell 3 — Sanity check(可選)

在啟動 Widget 之前,先用一個已知 GBM cell 跑一次,確認結果合理。
(預期:Top 推薦會集中在 PI3K/MTOR 路徑)

In [ ]:
from src.drug_ranking import predict_drug_ranking

# SIDM00083 = SF539 glioblastoma
ranking = predict_drug_ranking('SIDM00083', artifacts, top_k=10, with_ci=True)
print('Top 10 drugs for SF539 (GBM):')
print(ranking[['drug_name', 'P_sens', 'target', 'pathway']].to_string(index=False))

## Cell 4 — 啟動 Widget 🎮

執行完這個 cell 後,下方會出現:
- 🔽 **Cancer dropdown** — 選癌種(預設 Glioblastoma)
- 🔽 **Cell dropdown** — 該癌種下的 cell lines
- 🎚️ **Top K slider** — 5 到 30 的滑桿

選擇變更時會 **自動更新** 推薦表(每次 < 2 秒)。

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

# === 建 widgets ===
cancer_dropdown = widgets.Dropdown(
    options=sorted(cells_by_type.keys()),
    value='Glioblastoma' if 'Glioblastoma' in cells_by_type
          else sorted(cells_by_type.keys())[0],
    description='Cancer:',
)

initial_cells = cells_by_type[cancer_dropdown.value]
cell_dropdown = widgets.Dropdown(
    options=[f'{cid} ({name})' for cid, name in initial_cells],
    description='Cell:',
)

top_k_slider = widgets.IntSlider(
    value=10, min=5, max=30, step=5, description='Top K:',
)

output = widgets.Output()


def on_cancer_change(change):
    """當 cancer 改變時,更新 cell dropdown 的選項。"""
    ctype = change['new']
    cell_dropdown.options = [
        f'{cid} ({name})' for cid, name in cells_by_type[ctype]
    ]


def update_ranking(*args):
    """重新計算推薦並顯示。"""
    cell_id = cell_dropdown.value.split(' ')[0]
    top_k = top_k_slider.value

    with output:
        clear_output()
        try:
            ranking = predict_drug_ranking(
                cell_id, artifacts, top_k=top_k, with_ci=True
            )
            meta = artifacts.cell_metadata_lookup[cell_id]
            print('=' * 70)
            print(f'Cell: {cell_id} ({meta["CELL_LINE_NAME"]})')
            print(f'Cancer type: {meta["CANCER_TYPE"]}')
            print('=' * 70)

            display_df = ranking.copy()
            display_df['P_sens'] = display_df['P_sens'].apply(lambda x: f'{x:.3f}')
            display_df['CI'] = display_df.apply(
                lambda r: f'[{r["CI_lo"]:.3f}, {r["CI_hi"]:.3f}]', axis=1,
            )
            display_df['target'] = display_df['target'].apply(
                lambda t: str(t)[:25] + '..' if len(str(t)) > 27 else str(t)
            )
            display_df['pathway'] = display_df['pathway'].apply(
                lambda p: str(p)[:25] + '..' if len(str(p)) > 27 else str(p)
            )
            print(display_df[
                ['drug_name', 'P_sens', 'CI', 'target', 'pathway']
            ].to_string(index=False))
        except Exception as e:
            print(f'Error: {e}')


# 綁定 callback
cancer_dropdown.observe(on_cancer_change, names='value')
cancer_dropdown.observe(update_ranking, names='value')
cell_dropdown.observe(update_ranking, names='value')
top_k_slider.observe(update_ranking, names='value')

# 顯示 UI
ui = widgets.VBox([cancer_dropdown, cell_dropdown, top_k_slider, output])
display(ui)

# 初次顯示
update_ranking()
print('\n🎮 Widget ready — 在上方下拉選單互動!')

## Cell 5 — (選用)直接以 Python API 查詢

如果您不要 Widget,只想批次查詢,可以直接呼叫 `predict_drug_ranking()`:

In [ ]:
# 範例:對 3 種癌種各挑 1 個 cell,各取 Top 5 推薦
from itertools import islice

for ctype in ['Glioblastoma', 'Glioma', 'Neuroblastoma']:
    if ctype not in cells_by_type:
        continue
    cid, cname = next(islice(cells_by_type[ctype], 1))
    print(f'\n=== {ctype}: {cid} ({cname}) — Top 5 ===')
    top5 = predict_drug_ranking(cid, artifacts, top_k=5, with_ci=True)
    print(top5[['drug_name', 'P_sens', 'CI_lo', 'CI_hi', 'pathway']]
          .to_string(index=False))